SQL Databases - Parsing & Processing

In [1]:
import sqlite3
import os

os.makedirs("data/databases", exist_ok=True)

In [2]:
# create sample database
conn = sqlite3.connect("data/databases/company.db")
cursor  = conn.cursor()

In [3]:
# create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (id INTEGER PRIMARY KEY, name TEXT, role TEXT, department TEXT, salary REAL)""")

In [4]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS projects (id INTEGER PRIMARY KEY, name TEXT, status TEXT, budget TEXT, lead_id INTEGER)""")

In [5]:
# Insert Sample data in both tables
employees = [
    (1, "Alice", "Engineer", "R&D", 90000),
    (2, "Bob", "Manager", "Sales", 120000),
    (3, "Charlie", "Analyst", "Finance", 70000),
    (4, "David", "Engineer", "R&D", 95000)
]

projects = [
    (1, "Project Alpha", "Active", "100000", None),
    (2, "Project Beta", "Completed", "150000", 1),
    (3, "Project Gamma", "Active", "200000", 2),
    (4, "Project Delta", "Planned", "250000", 3)
]

In [6]:
cursor.executemany("""INSERT or REPLACE INTO employees (id, name, role, department, salary) VALUES (?, ?, ?, ?, ?)""", employees)
cursor.executemany("""INSERT or REPLACE INTO projects (id, name, status, budget, lead_id) VALUES (?, ?, ?, ?, ?)""", projects)

In [9]:
cursor.execute("""Select * from employees""").fetchall()

[(1, 'Alice', 'Engineer', 'R&D', 90000.0),
 (2, 'Bob', 'Manager', 'Sales', 120000.0),
 (3, 'Charlie', 'Analyst', 'Finance', 70000.0),
 (4, 'David', 'Engineer', 'R&D', 95000.0)]

In [10]:
conn.commit()
conn.close()

Database Content Extraction

In [11]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

e:\Ultimate RAG Bootcamp Using Langchain, LangGraph & Langsmith\RAGUDEMY\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# Method 1: SQLDATABASE Utility
db = SQLDatabase.from_uri("sqlite:///data/databases/company.db")

# get Database info
print(f"Tables: {db.get_usable_table_names()}")
print(f"\n Table DDL:")
print(db.get_table_info())

Tables: ['employees', 'projects']

 Table DDL:

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	Alice	Engineer	R&D	90000.0
2	Bob	Manager	Sales	120000.0
3	Charlie	Analyst	Finance	70000.0
*/


CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	budget TEXT, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
1	Project Alpha	Active	100000	None
2	Project Beta	Completed	150000	1
3	Project Gamma	Active	200000	2
*/


In [23]:
from typing import List
from langchain_core.documents import Document

print("Custom SQL Processing")

def sql_to_documents(db_path:str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Get table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    
    documents = []
    
    for table in tables:
        table_name = table[0]
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        
        # Create a document for the table schema
        schema_doc = Document(
            page_content=f"Table: {table_name}\nColumns: {', '.join([col[1] for col in columns])}",
            metadata={"source": f"{table_name}_schema"}
        )
        documents.append(schema_doc)
        
        # Create documents for each row in the table
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()
        
        table_content = f"Table: {table_name}\nColumns: {', '.join([col[1] for col in columns])}\n"
        table_content += f"Total Records: {len(rows)}\n\n"
        column_names = [col[1] for col in columns]
        table_content += "Sample Records:\n"
        for row in rows:
            record = dict(zip(column_names, row))
            table_content += f"{record}\n"
        
        doc = Document(
            page_content=table_content,
            metadata={"source": f"{table_name}_data", "num_records": len(rows), "table_name": table_name, "data_type": "table_data"}
        )
        documents.append(doc)

    # Strategy 2: Create Relationship Documents
    # Example: join employees and projects on lead_id to show which employee leads which project
    cursor.execute("""
    SELECT e.name AS employee_name, p.name AS project_name, p.status AS project_status
    FROM employees e
    JOIN projects p ON e.id = p.lead_id;
    """)

    relationships = cursor.fetchall()
    relationship_content = "Employee-Project Relationships:\n"
    for rel in relationships:
        relationship_content += f"{rel[0]} leads {rel[1]} (Status: {rel[2]})\n"

    relationship_doc = Document(
        page_content=relationship_content,
        metadata={"source": "employee_project_relationships", "data_type": "relationships", "query":"employee_project_join"}
    )
    documents.append(relationship_doc)

    conn.close()
    return documents

Custom SQL Processing


In [24]:
sql_to_documents("data/databases/company.db")

[Document(metadata={'source': 'employees_schema'}, page_content='Table: employees\nColumns: id, name, role, department, salary'),
 Document(metadata={'source': 'employees_data', 'num_records': 4, 'table_name': 'employees', 'data_type': 'table_data'}, page_content="Table: employees\nColumns: id, name, role, department, salary\nTotal Records: 4\n\nSample Records:\n{'id': 1, 'name': 'Alice', 'role': 'Engineer', 'department': 'R&D', 'salary': 90000.0}\n{'id': 2, 'name': 'Bob', 'role': 'Manager', 'department': 'Sales', 'salary': 120000.0}\n{'id': 3, 'name': 'Charlie', 'role': 'Analyst', 'department': 'Finance', 'salary': 70000.0}\n{'id': 4, 'name': 'David', 'role': 'Engineer', 'department': 'R&D', 'salary': 95000.0}\n"),
 Document(metadata={'source': 'projects_schema'}, page_content='Table: projects\nColumns: id, name, status, budget, lead_id'),
 Document(metadata={'source': 'projects_data', 'num_records': 4, 'table_name': 'projects', 'data_type': 'table_data'}, page_content="Table: project